# HQ-SAM 배경 교체 테스트

SAM2 결과와 HQ-SAM 결과를 같은 파이프라인에서 비교합니다. 기본값은 HQ-SAM을 사용하며, config/CLI 옵션으로 선택 방식을 바꿀 수 있습니다.

In [ ]:
from google.colab import drive, files
from pathlib import Path
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline')
assert (PROJECT_ROOT / 'configs/pipeline.yaml').is_file(), f'프로젝트를 찾을 수 없습니다: {PROJECT_ROOT}'
%cd $PROJECT_ROOT

In [ ]:
import os, subprocess, sys
PACKAGE_DIR = Path('/content/food-image-cleanup-packages')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(PACKAGE_DIR), '--prefer-binary', '-r', 'requirements-colab.txt'], check=True)
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV['PYTHONPATH'] = str(PACKAGE_DIR) + os.pathsep + RUNTIME_ENV.get('PYTHONPATH', '')
print('파이프라인/HQ-SAM 의존성 설치 완료')

In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.download_models', '--models', 'yolo', 'sam2', 'big-lama', 'openclip', 'birefnet', 'sana', 'grounding-dino', 'hq-sam'], check=True, env=RUNTIME_ENV)
DETECTOR_PROFILE = 'food_specialized'
assert (PROJECT_ROOT / 'models/sam2.1_s.pt').is_file()
assert (PROJECT_ROOT / 'models/best.pt').is_file()
assert (PROJECT_ROOT / 'models/hq-sam/config.json').is_file(), 'HQ-SAM snapshot is missing: models/hq-sam'
print('기본 모델 준비 완료. HQ-SAM은 로컬 models/hq-sam 스냅샷을 사용합니다.')

In [ ]:
import ipywidgets as widgets
from IPython.display import display

business_type_widget = widgets.Dropdown(options=['cafe', 'restaurant', 'bakery', 'dessert shop', 'bar', 'food truck'], value='restaurant', description='Business')
desired_mood_widget = widgets.Dropdown(options=['cozy and warm', 'modern and clean', 'premium and elegant', 'bright and fresh', 'rustic and natural'], value='premium and elegant', description='Mood')
composition_mode_widget = widgets.Dropdown(options=[('Keep original plate and food', 'preserve_original_plate'), ('Food only on generated plate', 'generated_plate')], value='preserve_original_plate', description='Mode')
hq_sam_enabled_widget = widgets.Checkbox(value=True, description='Use HQ-SAM')
hq_sam_selection_widget = widgets.Dropdown(options=[('Patch missing pieces only', 'patch_missing'), ('Auto by box coverage', 'box_coverage'), ('Prefer HQ-SAM for visual test', 'hq_sam'), ('Keep SAM2 only', 'sam2'), ('Use larger mask', 'larger_area')], value='patch_missing', description='HQ select')
hq_sam_model_widget = widgets.Text(value='models/hq-sam', description='HQ model')
display(business_type_widget, desired_mood_widget, composition_mode_widget, hq_sam_enabled_widget, hq_sam_selection_widget, hq_sam_model_widget)

In [ ]:
import json, shutil
from PIL import Image
from IPython.display import display
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one food image.'
source_path = Path(next(iter(uploaded)))
input_path = Path('data/input') / f'example{source_path.suffix.lower()}'
input_path.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(source_path), input_path)
metadata = {
    'business_type': business_type_widget.value,
    'desired_mood': desired_mood_widget.value,
    'food_category': 'food',
    'foreground_position': 'center_lower',
    'light_direction': 'left',
    'composition_mode': composition_mode_widget.value,
    'require_food_visible_mask': composition_mode_widget.value != 'generated_plate',
}
metadata_path = Path('data/input/example_metadata.json')
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metadata, ensure_ascii=False, indent=2))
display(Image.open(input_path))

In [ ]:
command = [sys.executable, '-m', 'scripts.run_background_replacement', '--input', str(input_path), '--metadata', str(metadata_path), '--enable-matting', '--enable-background-generator', '--detector-profile', DETECTOR_PROFILE]
if hq_sam_enabled_widget.value:
    command.extend(['--enable-hq-sam', '--hq-sam-selection-mode', hq_sam_selection_widget.value, '--hq-sam-model-id', hq_sam_model_widget.value.strip()])
print('실행 명령:', ' '.join(map(str, command)))
result = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, env=RUNTIME_ENV)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print('종료 코드:', result.returncode)

In [ ]:
from IPython.display import Image as DisplayImage
report_path = Path('data/reports') / f'{input_path.stem}_background_replacement_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
hq_stage = report.get('stages', {}).get('step_2b_hq_sam_segmentation', {})
print(json.dumps({'status': report.get('status'), 'hq_sam': hq_stage, 'debug_artifacts': report.get('debug_artifacts')}, ensure_ascii=False, indent=2))
for name, artifact_path in report.get('debug_artifacts', {}).items():
    if any(key in name for key in ['hq_sam', 'sam_structural', 'plate_mask', 'food_active', 'plate_edge_repair', 'final_composite']):
        artifact = Path(artifact_path)
        if artifact.is_file():
            print(name, artifact)
            display(DisplayImage(filename=str(artifact)))